In [ ]:
import tpx3_pipeline.tpx_pipe_ioc as tpi
import tpx3_pipeline.socket_listener as scl
import dask.dataframe as dd
import threading
import socket
from pathlib import Path
import numpy as np
import time
import pandas as pd
import tpx3awkward as tpx

In [2]:
print(dir(tpx.processing))

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'cluster', 'cluster_raw_df', 'convert_tpx3_file', 'convert_tpx3_files', 'convert_tpx3_files_parallel', 'corrections', 'decode_tpx3_binary', 'decoding', 'files', 'find_unmatched_tpx3_files', 'pipeline', 'raw_as_numpy', 'schemas']


In [3]:
trigger, out_q, file_q = tpi.test_boot()
path = Path.cwd() / "test_data"
files = []
for f in path.iterdir():
    if "tpx" in f.suffix:
        files.append(f)
print(files)

[pipeline]	 starting up
DAEMON: entering daemon routine
[pipeline]	 daemon processes deployed
[PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000000.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000001.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000002.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000003.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000004.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000005.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000006.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000007.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000008.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000009.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/test_data/rawAO1_000010.tpx3'), PosixPath('/home/mtaka/Documents/tpx_pipeline/t

In [4]:
def server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        print("[server]\t starting up")
        sock.bind((scl.HOST,scl.SERVAL))
        sock.listen()
        conn, addr = sock.accept()
        print("[server]\t connected, sending files")
        with conn:
            for f in files:
                a = np.fromfile(f, dtype="<u8")
                conn.sendall(a)

        print("[server]\t finished sending files, closing")


In [ ]:
t = threading.Thread(target=server)
t.start()
time.sleep(1)
trigger.set()

[server]	 starting up
[pipeline]	 Triggered: connecting...
[pipeline]	 connection attempt:  0
... sucessful
[server]	 connected, sending files


[socket]	 writing out sequence  0  to buffer  0
[worker]	 claimed sequence item  0  of size  9092504  with tail(?) length  0
[worker]	 Processing 1136563 ints
[socket]	 writing out sequence  1  to buffer  1
[worker]	 claimed sequence item  1  of size  9670120  with tail(?) length  0
[worker]	 Processing 1208765 ints
[socket]	 writing out sequence  2  to buffer  2
[worker]	 claimed sequence item  2  of size  9201144  with tail(?) length  0
[worker]	 Processing 1150143 ints
[socket]	 writing out sequence  3  to buffer  3
[worker]	 claimed sequence item  3  of size  10485760  with tail(?) length  0
[worker]	 Processing 1310720 ints
[socket]	 writing out sequence  4  to buffer  4
[worker]	 claimed sequence item  4  of size  10485760  with tail(?) length  0
[worker]	 Processing 1310720 ints
[server]	 finished sending files, closing
[socket]	 writing out sequence  5  to buffer  5
[socket]	 0 packet received... suspecting eot... awaiting next update
[socket]	 writing out sequence  6  to buffe

In [6]:
res = []
while not out_q.empty():
    ind, frame = out_q.get()
    res.append(frame)

total = pd.concat(res)
print(total.sort_values(by="t"))

                 t     xc     yc  ToT_max  ToT_sum  n
0           446481  246.0  486.0      300      300  1
1           447297  406.0  330.0      100      100  1
2           447299  407.0  330.0      125      125  1
3           452662  511.0  348.0      400      400  1
4           453064  211.0  228.0      125      125  1
...            ...    ...    ...      ...      ... ..
89145  12800609971   85.0  208.0      275      275  1
89146  12800613324   55.0   31.0      175      175  1
89147  12800614906   66.0  161.0      250      250  1
89148  12800615942   68.0   37.0      225      225  1
89149  12800619943  145.0  116.0      300      300  1

[7116197 rows x 6 columns]


In [7]:
file_res = []
while not file_q.empty():
    ind, file = file_q.get()
    file_res.append(file)

# total = pd.concat(res)
print(file_res)

[PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_0_2.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_0_1.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_0_6.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_0_4.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_0_0.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_0_3.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_0_5.parquet')]


In [8]:
target  = Path.cwd() / "data"
files = []
for file in target.iterdir():
    if "parquet" in file.suffix:
        files.append(file)
print(len(files))
print(files[:5])
df = dd.read_parquet(files)
print(df.compute())

6
[PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_-1_5.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_-1_1.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_-1_2.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_-1_0.parquet'), PosixPath('/home/mtaka/Documents/tpx_pipeline/data/buff_-1_-1_4.parquet')]
                   t     xc     yc  ToT_max  ToT_sum  n
0             446481  246.0  486.0      300      300  1
1             447297  406.0  330.0      100      100  1
2             447299  407.0  330.0      125      125  1
3             452662  511.0  348.0      400      400  1
4             453064  211.0  228.0      125      125  1
...              ...    ...    ...      ...      ... ..
1178068  12800609971   85.0  208.0      275      275  1
1178069  12800613324   55.0   31.0      175      175  1
1178070  12800614906   66.0  161.0      250      250  1
1178071  12800615942   68.0   37.0      225      225  1
117807

In [9]:
from pathlib import Path
a = Path("123/abc")

In [16]:
b = a.with_stem(a.stem+ str(uuid.uuid4()))
print(b)

123/abc6a7d2855-418d-4f46-8825-241b561eca02


In [8]:
import uuid
print(uuid.uuid6())

1f164d53-837f-6492-83b7-f8edfc474438
